In [12]:
#clear workspace
rm(list=ls())

In [76]:
library(ipumsr)
library(dplyr)
library(tidyr)
library(survey)
library(srvyr)
library(labelled)


Attaching package: ‘tidyr’




The following objects are masked from ‘package:Matrix’:

    expand, pack, unpack




In [14]:
#Reno <- read.csv(file='usa_00063.csv', header=TRUE)
#head(data)
#ddi <- read_ipums_ddi("usa_00064.xml")
reno1 <- read_ipums_micro(data_file = "usa_00078.dat", ddi = "usa_00078.xml")
reno2 <- read_ipums_micro(data_file = "usa_00079.dat", ddi = "usa_00079.xml")

Use of data from IPUMS USA is subject to conditions including that users should cite the data appropriately. Use command `ipums_conditions()` for more details.



Use of data from IPUMS USA is subject to conditions including that users should cite the data appropriately. Use command `ipums_conditions()` for more details.



In [18]:
data <- bind_rows(reno1, reno2)
#we use dplyr bind_rows
#merge is for horizontal (add columns), rbind is for equal columns vertically, we have VERSIONHIST, HISTID in reno1 (including full count) and MET2013, CBSERIAL in reno2 (original census serial)
#data <- data %>% filter(METAREA==672 | MET2013==39900)

Reno does not appear in the 1% samples prior to 1960, pulling full data for 1850-1950. Because this creates a large extract, data now filtered in retrieval. Retrieval logic does not allow combining multiple metro case selectors, stitching together two extracts vertically to accomodate filters for 1850-2000-METAREA and 2010-2024 MET2013.

Reno does not appear at all prior to 1960. It may be a point of interest that MSA / SMA as a unit of measurement was shifting in the 1960s.

Nevertheless we stitch all the data available together.

In [60]:
#glimpse(reno1)
#glimpse(reno2)
glimpse(data)

Rows: 23,049
Columns: 30
$ YEAR        <int> 1960, 1960, 1960, 1960, 1960, 1960, 1960, 1960, 1960, 1960…
$ SAMPLE      <int+lbl> 196002, 196002, 196002, 196002, 196002, 196002, 196002…
$ SERIAL      <dbl> 1649447, 1649447, 1649447, 1649447, 1649447, 1649448, 1649…
$ HHWT        <dbl> 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20…
$ CLUSTER     <dbl> 1.960016e+12, 1.960016e+12, 1.960016e+12, 1.960016e+12, 1.…
$ METAREA     <int+lbl> 672, 672, 672, 672, 672, 672, 672, 672, 672, 672, 672,…
$ METAREAD    <int+lbl> 6720, 6720, 6720, 6720, 6720, 6720, 6720, 6720, 6720, …
$ STRATA      <dbl> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA…
$ GQ          <int+lbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, …
$ PERNUM      <dbl> 1, 2, 3, 4, 5, 1, 2, 1, 2, 3, 4, 1, 2, 1, 2, 3, 4, 1, 2, 3…
$ PERWT       <dbl> 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20…
$ SEX         <int+lbl> 1, 2, 2, 2, 2, 1, 2, 1, 2, 2, 2, 1, 2, 1, 2, 2, 2, 1, …
$ AGE         <

Fix missing

In [24]:
#IpumsR import does not code R native (NA) missing values
sum(is.na(data$RACE))
sum(is.na(data$HISPAN))

[1] 0

[1] 0

In [50]:
#RACE is harmonized by IPUMS to not include missing values, it is inferred from other household members. The single multilevel race variable RACHSING is not available before 2000. However, HISPAN explicitly codes a missing value.


In [26]:
#data %>% group_by(HISPAN=haven::as_factor(HISPAN)) %>% summarize(n=sum(PERWT)) %>% mutate(pct = n/sum(n))
HISPAN <- lbl_na_if (data$HISPAN, ~ .val == 9)
data %>% group_by(HISPAN=haven::as_factor(HISPAN)) %>% summarize(n=sum(PERWT)) %>% mutate(pct = n/sum(n))

HISPAN,n,pct
<fct>,<dbl>,<dbl>
Not Hispanic,1545756,0.800187188
Mexican,298877,0.154718821
Puerto Rican,8761,0.004535282
Cuban,3316,0.001716584
Other,75033,0.038842123


In [27]:
data %>% group_by(RACE=haven::as_factor(RACE)) %>% summarize(n=sum(PERWT)) %>% mutate(pct = n/sum(n))

RACE,n,pct
<fct>,<dbl>,<dbl>
White,1492046,0.772383283
Black/African American,48610,0.025163803
American Indian or Alaska Native,32982,0.017073700
Chinese,18236,0.009440179
Japanese,7179,0.003716333
Other Asian or Pacific Islander,79037,0.040914863
"Other race, nec",128182,0.066355618
Two major races,115287,0.059680299
Three or more major races,10184,0.005271923


Recode using dplyr (Tidyverse according to rpubs.com).
We code two integrated race ethnicity variables, RACEETH1 is White-inclusive for Latino, meaning those Latinos selecting White race are coded White. RACEETH2 reverses this logic, coding those individuals responding White as Latino if they identified ethnicity as Hispanic.

In [49]:
data <- data %>%
    mutate(RACEETH1 = labelled(
            case_when(
                RACE >6 & RACE <=9 ~ 6,
                HISPAN >1 & HISPAN <5 ~ 3,
                RACE == 1 ~ 5,
                RACE >3 & RACE <7 ~ 4,
                RACE == 2 ~ 2,
                RACE == 3 ~ 1,
                .default = NA), 
            labels = c("American Indian" = 1, Black = 2, Latino = 3, Asian = 4, White = 5, Other = 6) 
        ))


In [49]:
data <- data %>%
    mutate(RACEETH2 = labelled(
            case_when(
                RACE >6 & RACE <=9 ~ 6,
                RACE == 1 ~ 5,
                RACE >3 & RACE <7 ~ 4,
                HISPAN >1 & HISPAN <5 ~ 3,
                RACE == 2 ~ 2,
                RACE == 3 ~ 1,
                .default = NA), 
            labels = c("American Indian" = 1, Black = 2, Latino = 3, Asian = 4, White = 5, Other = 6) 
        ))


In [56]:
#Summarize RACEETH1 and RACEETH2 across all samples in the dataset
data %>% group_by (RACEETH1=haven::as_factor(RACEETH1)) %>% summarize(n=sum(PERWT)) %>% mutate(pct= n/sum(n))
data %>% group_by (RACEETH2=haven::as_factor(RACEETH2)) %>% summarize(n=sum(PERWT)) %>% mutate(pct= n/sum(n))

RACEETH1,n,pct
<fct>,<dbl>,<dbl>
American Indian,30922,0.01600731
Black,47308,0.02448980
Latino,44112,0.02283534
Asian,103448,0.05355164
White,1452300,0.75180808
Other,253653,0.13130784


RACEETH2,n,pct
<fct>,<dbl>,<dbl>
White,1492046,0.772383283
Black,47308,0.024489800
American Indian,30922,0.016007305
Asian,104452,0.054071375
Other,253653,0.131307840
Latino,3362,0.001740397


Create DECADE as a single time variable / x-axis.

In [73]:
#val_labels(data$SAMPLE)
#data %>% count(YEAR)

In [83]:
#Recode IPUMS SAMPLE into DECADE
#ipums_val_labels(data$SAMPLE)
data <- data %>%
    mutate(DECADE = labelled( 
        case_when(
            (SAMPLE >= 185000 & SAMPLE < 186000) | YEAR == 1850 ~ 1,
            (SAMPLE >= 186000 & SAMPLE < 187000) | YEAR == 1860 ~ 2,
            (SAMPLE >= 187000 & SAMPLE < 188000) | YEAR == 1870 ~ 3,
            (SAMPLE >= 188000 & SAMPLE < 189000) | YEAR == 1880 ~ 4,
            (SAMPLE >= 189000 & SAMPLE < 190000) | YEAR == 1890 ~ 5,
            (SAMPLE >= 190000 & SAMPLE < 191000) | YEAR == 1900 ~ 6,
            (SAMPLE >= 191000 & SAMPLE < 192000) | YEAR == 1910 ~ 7,
            (SAMPLE >= 192000 & SAMPLE < 193000) | YEAR == 1920 ~ 8,
            (SAMPLE >= 193000 & SAMPLE < 194000) | YEAR == 1930 ~ 9,
            (SAMPLE >= 194000 & SAMPLE < 195000) | YEAR == 1940 ~ 10,
            (SAMPLE >= 195000 & SAMPLE < 196000) | YEAR == 1950 ~ 11,
            (SAMPLE >= 196000 & SAMPLE < 197000) | YEAR == 1960 ~ 12,
            (SAMPLE >= 197000 & SAMPLE < 198000) | YEAR == 1970 ~ 13,
            (SAMPLE >= 198000 & SAMPLE < 199000) | YEAR == 1980 ~ 14,
            (SAMPLE >= 199000 & SAMPLE < 200000) | YEAR == 1990 ~ 15,
            (SAMPLE >= 200000 & SAMPLE < 201000) | YEAR == 2000 ~ 16,
            (SAMPLE >= 201000 & SAMPLE < 202000) | YEAR == 2010 ~ 17,
            (SAMPLE >= 202000 & SAMPLE < 203000) | YEAR == 2020 ~ 18,
            .default = NA),
        labels = c("1850" = 1, "1860" = 2, "1870" = 3, "1880" = 4, "1890" = 5, "1900" = 6, "1910" = 7, "1920" = 8, "1930" = 9, "1940" = 10, "1950" = 11, "1960" = 12, "1970" = 13, "1980" = 14, "1990" = 15, "2000" = 16, "2010" = 17, "2020" = 18)
        ))


In [84]:
data %>% group_by (DECADE=haven::as_factor(DECADE)) %>% summarize(n=sum(PERWT)) %>% mutate(pct = n/sum(n))


DECADE,n,pct
<fct>,<dbl>,<dbl>
1960,84700,0.04384641
1980,190400,0.09856384
1990,255866,0.13245344
2010,893909,0.46274737
2020,506868,0.26238894


In [110]:
data %>%
  count((RACEETH1=haven::as_factor(RACEETH1)), DECADE, PERWT) %>%
  pivot_wider(names_from = DECADE, values_from = (n=sum(data$PERWT)), values_fill = 0)

ERROR: [1m[33mError[39m in `pivot_wider()`:[22m
[33m![39m Can't select columns past the end.
[1m[22m[36mℹ[39m Location 1931743 doesn't exist.
[36mℹ[39m There are only 4 columns.


In [107]:
data %>%
  count((RACEETH1=haven::as_factor(RACEETH1)), (DECADE=haven::as_factor(DECADE))) %>%
  pivot_wider(names_from = (DECADE=haven::as_factor(DECADE)), values_from = n, values_fill = 0)

ERROR: [1m[33mError[39m in `pivot_wider()`:[22m
[1m[22m[36mℹ[39m In argument: `DECADE = haven::as_factor(DECADE)`.
[1mCaused by error in `haven::as_factor()`:[22m
[33m![39m object 'DECADE' not found


In [93]:
#crosstab over time
data %>% group_by(DECADE=haven::as_factor(DECADE)) %>%
    tally() %>%
    pivot_wider (data$DECADE, (RACEETH1=haven::as_factor(RACEETH1)))

ERROR: [1m[33mError[39m in `pivot_wider()`:[22m
[33m![39m Can't select columns that don't exist.
[31m✖[39m Column `name` doesn't exist.


In [8]:
#save(Reno_clean, file="Reno.RData")
saveRDS(data, file = "Reno.rds")

Code appendix

In [28]:
#ipums_val_labels(data$RACE)
#ipums_var_label(data$RACE)
#ipums_var_info(data$RACE)

In [29]:
#lbl_na_if()
#lbl_clean()
#as_factor()
#zap_labels()
#zap_ipums_attributes()

In [ ]:
#ipums_val_labels(data$RACE)
#ipums_val_labels(data$HISPAN)

In [ ]:
#data <- data %>%
#  mutate(status = labelled(
#    case_match(status, 
#      1 ~ 1, 
#      2 ~ 0,
#      .default = status
#    ),
#    labels = c(Active = 1, Inactive = 0) # Specify your new label mappings
#  ))
